# Ophiolite Log-to-AVO Golden Path

This notebook now uses the domain-first AVO surface rather than the workflow runner as the main modeling entrypoint.

- project/LAS setup stays in a short preamble
- the main modeling cells use `ElasticChannelBindings`, `LayeringSpec`, `AngleSampling`, and `AvoExperiment`


In [36]:
from importlib import import_module, reload
from pathlib import Path
import json
import sys

from IPython.display import display

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "Cargo.toml").exists() and (candidate / "python" / "src").exists():
            return candidate
    raise RuntimeError("could not locate the Ophiolite repo root from the current working directory")

REPO_ROOT = find_repo_root(Path.cwd())
PYTHON_SRC = REPO_ROOT / "python" / "src"
JUPYTER_PYTHON_SRC = REPO_ROOT / "python" / "jupyter" / "src"
NOTEBOOK_DIR = REPO_ROOT / "examples" / "golden_paths" / "log_avo"
for path in reversed((PYTHON_SRC, JUPYTER_PYTHON_SRC, NOTEBOOK_DIR)):
    normalized = str(path)
    if normalized in sys.path:
        sys.path.remove(normalized)
    sys.path.insert(0, normalized)

for module_name in tuple(sys.modules):
    if module_name == "log_avo_workflow" or module_name.startswith(("ophiolite_sdk", "ophiolite_automation", "ophiolite_jupyter")):
        del sys.modules[module_name]

workflow = reload(import_module("log_avo_workflow"))
detect_real_las_paths = workflow.detect_real_las_paths
load_demo_project = workflow.load_demo_project
AngleSampling = workflow.AngleSampling
AvoExperiment = workflow.AvoExperiment
ElasticChannelBindings = workflow.ElasticChannelBindings
LayeringSpec = workflow.LayeringSpec

from ophiolite_jupyter import AvoInterceptGradientCrossplotWidget, AvoResponseWidget

REPO_ROOT


PosixPath('/Users/sc/dev/ophiolite')

## Data selection and setup

The setup section decides whether to use the real F2-A5 LAS pair or the synthetic project fixture. After this cell, the notebook switches to domain nouns.

In [37]:
dsi_path, density_path = detect_real_las_paths()
data_mode = "real" if dsi_path is not None and density_path is not None else "synthetic"
layout, project, wellbore, data_summary = load_demo_project(data_mode=data_mode)
{
    "data_mode": data_mode,
    "project_root": str(project.root),
    "wellbore": wellbore.name,
    "data_summary": data_summary,
}

{'data_mode': 'real',
 'project_root': '/var/folders/__/63xydj956wb3x3734qw6cgxc0000gq/T/ophiolite_log_avo_5_8zrdxf/project',
 'wellbore': 'F2-A5',
 'data_summary': {'data_mode': 'real',
  'dsi_las_path': '/Users/sc/Downloads/SubsurfaceData/blocks/F3/F02_wells_data/F02-A-05/f02a05_20030103-1-06_a5_main_log_dsi_030lup.las',
  'density_las_path': '/Users/sc/Downloads/SubsurfaceData/blocks/F3/F02_wells_data/F02-A-05/f02a05_20030103-1-06_lwd2259a5rt.las',
  'tops_source_path': '/Users/sc/Downloads/SubsurfaceData/blocks/F3/F02_wells_data/F02-A-05/lithostratigrafie.txt',
  'tops_import': {'source_path': '/Users/sc/Downloads/SubsurfaceData/blocks/F3/F02_wells_data/F02-A-05/lithostratigrafie.txt',
   'source_name': 'EINDSLIP',
   'reported_well_name': 'F02-A-05',
   'reported_depth_reference': 'Kelly Bushing',
   'resolved_source_depth_reference': 'Kelly Bushing',
   'resolved_depth_domain': 'md',
   'resolved_depth_datum': 'kb',
   'source_row_count': 19,
   'imported_row_count': 18,
   'omit

## Canonical elastic bindings

This is the first domain-facing modeling choice: which canonical elastic channel types should drive the AVO workflow.

In [38]:
bindings = ElasticChannelBindings(
    vp="Dt",
    vs="Dts",
    density="Rho",
)
elastic = wellbore.elastic_log_set(bindings=bindings)
top_set = wellbore.top_set("lithostrat-tops") or wellbore.top_set()
selected_top_set_selectors = (
    top_set.interval_selectors[7:9]
    if top_set is not None and len(top_set.interval_selectors) >= 9
    else top_set.interval_selectors[:2]
    if top_set is not None
    else ()
)
{
    "available_log_types": wellbore.available_log_types(),
    "available_top_sets": wellbore.available_top_sets(),
    "available_marker_sets": wellbore.available_marker_sets(),
    "top_set_asset_name": None if top_set is None else top_set.asset_name,
    "top_set_interval_selectors": () if top_set is None else top_set.interval_selectors,
    "selected_top_set_selectors": selected_top_set_selectors,
    "vp": {
        "source_curve": elastic.vp.source_curve.curve_name,
        "source_semantic_type": elastic.vp.source_semantic_type,
        "source_log_type": elastic.vp.source_curve.log_type,
        "derivation": elastic.vp.derivation,
    },
    "vs": {
        "source_curve": elastic.vs.source_curve.curve_name,
        "source_semantic_type": elastic.vs.source_semantic_type,
        "source_log_type": elastic.vs.source_curve.log_type,
        "derivation": elastic.vs.derivation,
    },
    "density": {
        "source_curve": elastic.density.source_curve.curve_name,
        "source_semantic_type": elastic.density.source_semantic_type,
        "source_log_type": elastic.density.source_curve.log_type,
        "derivation": elastic.density.derivation,
    },
}

{'available_log_types': ['BulkDensity',
  'CompressionalSlowness',
  'GammaRay',
  'NeutronPorosity',
  'ShearSlowness'],
 'available_top_sets': ['lithostrat-tops'],
 'available_marker_sets': [],
 'top_set_asset_name': 'lithostrat-tops',
 'top_set_interval_selectors': ('NUMS',
  'NUOT',
  'NUBA',
  'NMRF',
  'NLFF',
  'NLFFT',
  'NLLF#1',
  'NLLFC#1',
  'CKEK#1',
  'NLLFC#2',
  'CKEK#2',
  'NLLF#2',
  'CKEK#3',
  'NLLF#3',
  'CKEK#4',
  'NLLF#4',
  'NLLFC#3',
  'CKEK#5'),
 'selected_top_set_selectors': ('NLLFC#1', 'CKEK#1'),
 'vp': {'source_curve': 'DTCO',
  'source_semantic_type': 'Sonic',
  'source_log_type': 'CompressionalSlowness',
  'derivation': 'sonic_to_vp'},
 'vs': {'source_curve': 'DTSM',
  'source_semantic_type': 'ShearSonic',
  'source_log_type': 'ShearSlowness',
  'derivation': 'shear_sonic_to_vs'},
 'density': {'source_curve': 'BDCX',
  'source_semantic_type': 'BulkDensity',
  'source_log_type': 'BulkDensity',
  'derivation': None}}

## Layering and experiment specification

These are the core QuantLib-like spec objects for the public AVO surface. The fixed-interval spec stays available, but when a top set exists the notebook switches to selector-based interval layering without dropping to raw rows.

In [39]:
fixed_layering = LayeringSpec.fixed_interval(20, unit="ft")
top_set_layering = (
    top_set.layering(selectors=selected_top_set_selectors)
    if top_set is not None and selected_top_set_selectors
    else None
)
layering = top_set_layering or fixed_layering
experiment = AvoExperiment.zoeppritz(
    angles=AngleSampling.range(0, 40, 5)
)
{
    "fixed_layering": fixed_layering,
    "top_set_layering": top_set_layering,
    "active_layering": layering,
    "experiment": experiment,
}

{'fixed_layering': LayeringSpec(kind='fixed_interval', thickness_value=20.0, unit='ft', depth_edges=(), labels=(), selectors=(), asset_name=None, min_samples_per_layer=3, include_partial_last_layer=False, allow_gaps=False),
 'top_set_layering': LayeringSpec(kind='top_set', thickness_value=None, unit='m', depth_edges=(), labels=(), selectors=('NLLFC#1', 'CKEK#1'), asset_name='lithostrat-tops', min_samples_per_layer=3, include_partial_last_layer=False, allow_gaps=True),
 'active_layering': LayeringSpec(kind='top_set', thickness_value=None, unit='m', depth_edges=(), labels=(), selectors=('NLLFC#1', 'CKEK#1'), asset_name='lithostrat-tops', min_samples_per_layer=3, include_partial_last_layer=False, allow_gaps=True),
 'experiment': AvoExperiment(method='zoeppritz', angles=AngleSampling(values=(0.0, 5.0, 10.0, 15.0, 20.0, 25.0, 30.0, 35.0, 40.0)))}

## Run the AVO experiment

The main execution step now reads as `elastic.run_avo(...)`, with the layering chosen from stable interval selectors when a top set is available.

In [40]:
result = elastic.run_avo(
    layering=layering,
    experiment=experiment,
)
{
    "layer_count": len(result.layers),
    "interface_count": len(result.interfaces),
    "first_interface_depth_m": result.interfaces[0].depth_m if result.interfaces else None,
}


{'layer_count': 2, 'interface_count': 1, 'first_interface_depth_m': 1883.5}

## Chart-ready outputs

The result object owns the handoff into the AVO chart family. The response curves come from the `zoeppritz` experiment, while the intercept-gradient crossplot uses a `shuey_two_term` experiment because that surface needs intercept and gradient attributes.

In [41]:
response_source = result.response_source(
    title=f"{wellbore.name} Zoeppritz AVO",
    subtitle="Selector-driven top-set layering" if top_set_layering is not None else "Fixed-interval layering",
)
crossplot_result = elastic.run_avo(
    layering=layering,
    experiment=AvoExperiment.shuey_two_term(
        angles=AngleSampling.range(0, 40, 5)
    ),
)
crossplot_source = crossplot_result.crossplot_source(
    title=f"{wellbore.name} Intercept-Gradient Crossplot",
    subtitle=(
        "Selector-driven top-set layering (Shuey two-term)"
        if top_set_layering is not None
        else "Fixed-interval layering (Shuey two-term)"
    ),
)

response_path = layout.output_root / "notebook_avo_response_source.json"
crossplot_path = layout.output_root / "notebook_avo_crossplot_source.json"
response_path.write_text(json.dumps(response_source, indent=2) + "\n")
crossplot_path.write_text(json.dumps(crossplot_source, indent=2) + "\n")
{
    "response_path": str(response_path),
    "crossplot_path": str(crossplot_path),
}


{'response_path': '/var/folders/__/63xydj956wb3x3734qw6cgxc0000gq/T/ophiolite_log_avo_5_8zrdxf/outputs/notebook_avo_response_source.json',
 'crossplot_path': '/var/folders/__/63xydj956wb3x3734qw6cgxc0000gq/T/ophiolite_log_avo_5_8zrdxf/outputs/notebook_avo_crossplot_source.json'}

In [42]:
response_source

{'schema_version': 1,
 'id': 'wellbore_1776794277189082000_2-avo-response',
 'name': 'F2-A5 AVO',
 'title': 'F2-A5 Zoeppritz AVO',
 'subtitle': 'Selector-driven top-set layering',
 'x_axis': {'label': 'Incidence Angle', 'unit': 'deg'},
 'y_axis': {'label': 'PP Reflectivity', 'unit': 'ratio'},
 'interfaces': [{'id': 'interface-1',
   'label': 'Interface 1',
   'color': '#0B7285',
   'reservoir_label': '1883.5 m'}],
 'series': [{'id': 'interface-1-series',
   'interface_id': 'interface-1',
   'label': 'Interface 1',
   'color': '#0B7285',
   'style': 'solid',
   'reflectivity_model': 'zoeppritz',
   'anisotropy_mode': 'isotropic',
   'incidence_angles_deg': [0.0,
    5.0,
    10.0,
    15.0,
    20.0,
    25.0,
    30.0,
    35.0,
    40.0],
   'values': [-0.11064784228801727,
    -0.1123727485537529,
    -0.11755826324224472,
    -0.1262391209602356,
    -0.13848116993904114,
    -0.15439367294311523,
    -0.17414703965187073,
    -0.1979968100786209,
    -0.22631408274173737]}]}

## Notebook widgets

These widgets keep the notebook as the Python host while the existing Ophiolite Charts AVO wrappers remain the renderer.


In [43]:
response_widget = AvoResponseWidget.from_result(
    result,
    height_px=420,
    title=response_source["title"],
    subtitle=response_source.get("subtitle"),
)
crossplot_widget = AvoInterceptGradientCrossplotWidget.from_result(
    crossplot_result,
    height_px=420,
    title=crossplot_source["title"],
    subtitle=crossplot_source.get("subtitle"),
)
display(response_widget)
display(crossplot_widget)
response_widget.fit_to_data()
crossplot_widget.fit_to_data()
{
    "response_fit_request_id": response_widget.fit_request_id,
    "crossplot_fit_request_id": crossplot_widget.fit_request_id,
    "response_frontend_error": response_widget.frontend_error,
    "crossplot_frontend_error": crossplot_widget.frontend_error,
}


{'response_fit_request_id': 1,
 'crossplot_fit_request_id': 1,
 'response_frontend_error': None,
 'crossplot_frontend_error': None}